# Module 3 — Response Evaluation

**Conversational Emotional Trajectory Modeling & Adaptive Dialogue System**

---

## What this notebook does

This notebook implements **Section C: Response Evaluation** from the project proposal. It answers the central hypothesis:

> *Does an emotion-aware dialogue system produce responses that are at least 20% better (in empathy, appropriateness, and helpfulness) than a baseline LLM?*

## What changed in this version

Following the same pattern used in the Module 2 notebook (`emotional_trajectory_tracker.ipynb` v2), this notebook now:

1. **Loads the real fine-tuned RoBERTa model** from `module1_model/` (safetensors) — no more manually specifying emotion scores.
2. **Runs the full Module 1 → Module 2 → Module 3 pipeline** on raw text conversations.
3. **Includes the `predict_emotions()` and `run_full_pipeline()` helpers** that Module 2 defined, so you can pass an entire conversation as a list of strings.
4. Keeps the simulated-pair pipeline as a fallback for testing without the model.

## Module scope & dependencies

| Module | Status in this notebook | Purpose |
|---|---|---|
| **Module 1** — RoBERTa emotion classifier | ✅ Loaded from safetensors | Produces per-turn emotion scores from raw text |
| **Module 2** — Emotional Trajectory Tracker | ✅ Replicated inline | Computes the conditioning prompt |
| **Module 3** — Response Evaluation | ✅ Full implementation | The focus of this notebook |

## Pipeline

```
Raw text conversation (list[str])
         ↓
Module 1 (RoBERTa → per-turn emotion scores)
         ↓
Module 2 (trajectory signals → conditioning prompt)
         ↓
     ┌───┴────┐
     ↓        ↓
  Baseline  Conditioned    ← two LLM responses per user turn
     ↓        ↓
     └───┬────┘
         ↓
  Automated scoring (BERTScore + Empathy + Specificity)
         ↓
  Human rater CSV (for 20–30 evaluators, optional)
         ↓
  Final report + CSV export
```


## 0. Install Dependencies

**Required:** `numpy`, `pandas`, `tqdm`, `torch`, `transformers`, `safetensors`

**Recommended:** `bert-score` (for real semantic scoring — otherwise a fallback is used)

**Optional:** `openai` (only if you want to generate real LLM responses instead of using simulated examples)


In [ ]:
# Uncomment and run if needed
# !pip install numpy pandas tqdm
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# !pip install transformers safetensors
# !pip install bert-score
# !pip install openai  # optional, for live API generation


## 1. Imports & Emotion Taxonomy (from Module 2)

**Why this block exists:** Module 3 needs to call `build_conditioning_prompt()` from Module 2. To keep this notebook runnable standalone, we replicate the minimum Module 2 code needed.

**What's happening:**
- `EMOTION_CLUSTERS` maps fine-grained RoBERTa labels (e.g. `anxiety`, `stress`, `self-doubt`) to 7 core Ekman clusters (Joy, Sadness, Anger, Fear, Disgust, Surprise, Contempt)
- `VALENCE` (−1 to +1) encodes positive/negative affect
- `AROUSAL` (0 to 1) encodes activation/energy level


In [21]:
from __future__ import annotations

import math
import json
import csv
import os
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

# Emotion taxonomy (same as Module 2)
EMOTION_CLUSTERS: Dict[str, str] = {
    "joy": "Joy", "happiness": "Joy", "excitement": "Joy", "amusement": "Joy",
    "gratitude": "Joy", "love": "Joy", "pride": "Joy", "relief": "Joy", "optimism": "Joy",
    "admiration": "Joy", "approval": "Joy", "caring": "Joy", "desire": "Joy",
    "sadness": "Sadness", "grief": "Sadness", "disappointment": "Sadness",
    "loneliness": "Sadness", "melancholy": "Sadness", "remorse": "Sadness",
    "anger": "Anger", "frustration": "Anger", "annoyance": "Anger",
    "rage": "Anger", "resentment": "Anger", "disapproval": "Anger",
    "fear": "Fear", "anxiety": "Fear", "nervousness": "Fear", "stress": "Fear",
    "worry": "Fear", "panic": "Fear", "dread": "Fear", "apprehension": "Fear",
    "self-doubt": "Fear", "pressure": "Fear", "overwhelm": "Fear", "embarrassment": "Fear",
    "disgust": "Disgust", "revulsion": "Disgust",
    "surprise": "Surprise", "shock": "Surprise", "amazement": "Surprise",
    "confusion": "Surprise", "curiosity": "Surprise", "realization": "Surprise",
    "contempt": "Contempt", "disdain": "Contempt", "scorn": "Contempt",
    "neutral": "Neutral",
}

VALENCE: Dict[str, float] = {
    "Joy": +1.0, "Surprise": +0.2,
    "Sadness": -0.7, "Fear": -0.8, "Anger": -0.9,
    "Disgust": -0.85, "Contempt": -0.6, "Neutral": 0.0,
}

AROUSAL: Dict[str, float] = {
    "Joy": 0.7, "Surprise": 0.9,
    "Sadness": 0.3, "Fear": 0.85, "Anger": 0.95,
    "Disgust": 0.6, "Contempt": 0.5, "Neutral": 0.2,
}

print("Imports complete. Taxonomy loaded:", list(VALENCE.keys()))


Imports complete. Taxonomy loaded: ['Joy', 'Surprise', 'Sadness', 'Fear', 'Anger', 'Disgust', 'Contempt', 'Neutral']


## 2. Data Containers (from Module 2)

**`TurnSnapshot`** holds one utterance's emotional state after clustering. The `__post_init__` method:
1. Maps raw labels to clusters (e.g. `"anxiety" + "stress" → Fear`)
2. Normalises cluster weights so they sum to 1
3. Computes weighted valence and arousal (used downstream for momentum and escalation)

**`TrajectorySignals`** is the final output of Module 2 — all the higher-order signals. Module 3 converts this into a conditioning prompt.


In [22]:
@dataclass
class TurnSnapshot:
    """One utterance's emotional state after clustering."""
    turn_idx: int
    raw_emotions: Dict[str, float]
    clustered: Dict[str, float] = field(default_factory=dict)
    dominant: str = ""
    intensity: float = 0.0
    valence: float = 0.0
    arousal: float = 0.0

    def __post_init__(self):
        acc: Dict[str, float] = defaultdict(float)
        for label, score in self.raw_emotions.items():
            cluster = EMOTION_CLUSTERS.get(label.lower(), label.title())
            acc[cluster] += score

        total = sum(acc.values()) or 1.0
        self.clustered = {k: v / total for k, v in acc.items()}
        self.dominant = max(self.clustered, key=self.clustered.get)
        self.intensity = float(np.mean(list(self.raw_emotions.values())))

        v = a = w = 0.0
        for cluster, weight in self.clustered.items():
            v += VALENCE.get(cluster, 0.0) * weight
            a += AROUSAL.get(cluster, 0.0) * weight
            w += weight
        if w:
            self.valence = round(v / w, 4)
            self.arousal = round(a / w, 4)


@dataclass
class TrajectorySignals:
    """Full output of Module 2 — all higher-order signals."""
    turns: int = 0
    emotional_momentum: float = 0.0
    volatility_index: float = 0.0
    escalation_score: float = 0.0
    dominant_state: str = ""
    transition_matrix: Dict[str, Dict[str, float]] = field(default_factory=dict)
    emotion_sequence: List[str] = field(default_factory=list)
    intensity_series: List[float] = field(default_factory=list)
    valence_series: List[float] = field(default_factory=list)
    arousal_series: List[float] = field(default_factory=list)
    snapshots: List[TurnSnapshot] = field(default_factory=list)

print("Data containers defined.")


Data containers defined.


## 3. Emotional Trajectory Tracker & Conditioning Prompt (from Module 2)

Three key computations:
1. **Momentum** — exponentially-weighted mean of valence differences
2. **Volatility** — standard deviation of intensity, normalised
3. **Escalation** — regression slope of negative-affect activation

Then `build_conditioning_prompt()` translates these numbers into a natural-language instruction for the LLM.


In [23]:
class EmotionalTrajectoryTracker:
    """Module 2 — tracks emotional signals across conversation turns."""

    def __init__(self, window: int = 3):
        self.window = window
        self._snapshots: List[TurnSnapshot] = []

    def add_turn(self, emotions: Dict[str, float]) -> TurnSnapshot:
        snap = TurnSnapshot(
            turn_idx=len(self._snapshots),
            raw_emotions={k: float(v) for k, v in emotions.items()}
        )
        self._snapshots.append(snap)
        return snap

    def reset(self):
        self._snapshots.clear()

    def compute(self) -> TrajectorySignals:
        n = len(self._snapshots)
        if n == 0:
            raise ValueError("No turns added.")

        s = TrajectorySignals(turns=n, snapshots=list(self._snapshots))
        s.emotion_sequence = [x.dominant for x in self._snapshots]
        s.intensity_series = [x.intensity for x in self._snapshots]
        s.valence_series = [x.valence for x in self._snapshots]
        s.arousal_series = [x.arousal for x in self._snapshots]

        cluster_scores: Dict[str, float] = defaultdict(float)
        for i, snap in enumerate(self._snapshots):
            w = (i + 1) / n
            for cluster, score in snap.clustered.items():
                cluster_scores[cluster] += score * w
        s.dominant_state = max(cluster_scores, key=cluster_scores.get)

        # Momentum
        if len(s.valence_series) >= 2:
            diffs = [s.valence_series[i] - s.valence_series[i-1] for i in range(1, n)]
            weights = [math.exp(i) for i in range(len(diffs))]
            w_total = sum(weights)
            s.emotional_momentum = float(np.clip(
                sum(d*w for d, w in zip(diffs, weights)) / w_total, -1, 1
            ))

        # Volatility
        if n >= 2:
            s.volatility_index = float(np.clip(np.std(s.intensity_series) / 0.5, 0, 1))

        # Escalation
        if n >= 2:
            neg_act = [ar * (1 - (vl + 1) / 2)
                       for vl, ar in zip(s.valence_series, s.arousal_series)]
            slope, _ = np.polyfit(np.arange(n, dtype=float), neg_act, 1)
            s.escalation_score = float(np.clip(slope * n, -1, 1))

        # Transition matrix
        states = sorted(set(s.emotion_sequence))
        counts = {st: {t: 0.0 for t in states} for st in states}
        for a, b in zip(s.emotion_sequence, s.emotion_sequence[1:]):
            counts[a][b] += 1.0
        for src in states:
            total = sum(counts[src].values())
            if total:
                for tgt in states:
                    counts[src][tgt] /= total
        s.transition_matrix = counts
        return s


def build_conditioning_prompt(signals: TrajectorySignals) -> str:
    """Translate Module 2 signals into an LLM system prompt."""
    esc = signals.escalation_score
    vol = signals.volatility_index
    trajectory_str = " → ".join(signals.emotion_sequence)

    traj_prefix = ("escalating" if esc > 0.15
                   else "de-escalating" if esc < -0.15
                   else "stable")

    vol_label = ("very high" if vol > 0.6
                 else "high" if vol > 0.35
                 else "moderate" if vol > 0.15
                 else "low")

    if esc > 0.2 and vol > 0.3:
        tone = "validation + grounding + emotional de-escalation"
    elif esc > 0.1:
        tone = "empathetic acknowledgement + gentle reframing"
    elif esc < -0.1:
        tone = "reinforcing + warm encouragement"
    elif vol > 0.4:
        tone = "stabilising + calm, consistent reassurance"
    else:
        tone = "neutral + supportive"

    return (
        f"[EMOTIONAL CONTEXT — Module 2 output]\n"
        f"User emotional trajectory: {traj_prefix} {trajectory_str.lower()}.\n"
        f"Dominant state: {signals.dominant_state}. "
        f"Momentum: {signals.emotional_momentum:+.3f}. "
        f"Volatility: {vol_label} ({signals.volatility_index:.3f}). "
        f"Escalation: {signals.escalation_score:+.3f}.\n"
        f"Recommended tone: {tone}.\n"
        f"[END EMOTIONAL CONTEXT]"
    )

print("Tracker and conditioning prompt builder defined.")


Tracker and conditioning prompt builder defined.


## 4. Module 1 Model — Load from Safetensors & Run Inference

This is the **key update** over the previous Module 3 version: instead of hardcoding emotion scores, we now load the fine-tuned RoBERTa model saved from Module 1 training and run it on raw utterance strings.

**Your saved directory contains all required files:**
```
module1_model/
  model.safetensors     ← fine-tuned weights (~487 MB)
  config.json           ← model architecture config
  tokenizer.json        ← fast tokenizer
  tokenizer_config.json ← tokenizer settings
  vocab.json            ← RoBERTa BPE vocabulary
  merges.txt            ← RoBERTa BPE merge rules
  special_tokens_map.json
  training_args.bin     ← saved Trainer hyperparameters (not needed for inference)
```
All files are loaded from disk — **no internet connection required**.

> If you want to skip model loading and use the simulated mode (Section 7), set `USE_REAL_MODEL = False` in the next cell.


In [ ]:
# ── Toggle: set to False to skip model loading and use simulated mode only ────
USE_REAL_MODEL = True

# ── Path to your Module 1 saved model directory ───────────────────────────────
MODEL_DIR = Path(
    r"/Users/kunalgupta/Downloads/conversational-emotion-trajectory-mvp-main/outputs/module1_goemotions/module1_model"
)

# Sigmoid threshold for multi-label output (same default as Module 2 notebook)
INFERENCE_THRESHOLD = 0.30

tokenizer = None
model = None
GOEMOTION_LABELS: List[str] = []
device = None

if USE_REAL_MODEL:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    # ── 1. Verify all expected files are present ──────────────────────────────
    REQUIRED_FILES = [
        "model.safetensors", "config.json", "tokenizer.json",
        "tokenizer_config.json", "vocab.json", "merges.txt",
        "special_tokens_map.json",
    ]
    print("Checking model directory:")
    all_present = True
    for fname in REQUIRED_FILES:
        fpath = MODEL_DIR / fname
        status = "✓" if fpath.exists() else "✗ MISSING"
        size = f"({fpath.stat().st_size / 1024 / 1024:.1f} MB)" if fpath.exists() else ""
        print(f"  {status}  {fname} {size}")
        if not fpath.exists():
            all_present = False

    if not all_present:
        print("\n  ⚠️ One or more files missing. Falling back to simulated mode.")
        USE_REAL_MODEL = False

if USE_REAL_MODEL:
    # ── 2. Read label mapping from config.json ────────────────────────────────
    with open(MODEL_DIR / "config.json") as f:
        _cfg = json.load(f)

    if "id2label" in _cfg:
        GOEMOTION_LABELS = [_cfg["id2label"][str(i)] for i in range(len(_cfg["id2label"]))]
        print(f"\nLabel mapping loaded from config.json — {len(GOEMOTION_LABELS)} labels.")
    else:
        # Fallback: standard GoEmotions 28-label order
        GOEMOTION_LABELS = [
            "admiration", "amusement", "anger", "annoyance", "approval", "caring",
            "confusion", "curiosity", "desire", "disappointment", "disapproval",
            "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
            "joy", "love", "nervousness", "optimism", "pride", "realization",
            "relief", "remorse", "sadness", "surprise", "neutral",
        ]
        print(f"\nNo id2label in config — using default GoEmotions order ({len(GOEMOTION_LABELS)} labels).")
    print("  Labels:", GOEMOTION_LABELS)

    # ── 3. Load tokenizer + model ─────────────────────────────────────────────
    print(f"\nLoading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
    print(f"  ✓ Tokenizer loaded  (vocab size: {tokenizer.vocab_size:,})")

    print(f"\nLoading model weights from model.safetensors...")
    model = AutoModelForSequenceClassification.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
    )
    print(f"  ✓ Weights loaded")

    # ── 4. Move to device and set eval mode ───────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    print(f"\n{'─'*40}")
    print(f"  Model     : {model.config.model_type} for sequence classification")
    print(f"  Labels    : {model.config.num_labels}")
    print(f"  Device    : {device}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'─'*40}")
else:
    print("\nSkipping model load — Section 7 (simulated pairs) will be used instead.")


Checking model directory:
  ✓  model.safetensors (475.5 MB)
  ✓  config.json (0.0 MB)
  ✓  tokenizer.json (3.4 MB)
  ✓  tokenizer_config.json (0.0 MB)
  ✓  vocab.json (0.8 MB)
  ✓  merges.txt (0.4 MB)
  ✓  special_tokens_map.json (0.0 MB)

Label mapping loaded from config.json — 7 labels.
  Labels: ['joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise', 'contempt']

Loading tokenizer...
  ✓ Tokenizer loaded  (vocab size: 50,265)

Loading model weights from model.safetensors...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7460.99it/s]

  ✓ Weights loaded

────────────────────────────────────────
  Model     : roberta for sequence classification
  Labels    : 7
  Device    : cpu
  Parameters: 124,651,015
────────────────────────────────────────


### 4.1 `predict_emotions()` — Run Module 1 on a list of strings

This is the same helper used in the Module 2 notebook. Given a list of utterance strings, it returns a list of emotion dicts — one per turn, ready to feed into `EmotionalTrajectoryTracker.add_turn()`.


In [25]:
def predict_emotions(
    utterances: List[str],
    threshold: float = INFERENCE_THRESHOLD,
    batch_size: int = 8,
) -> List[Dict[str, float]]:
    """
    Run the Module 1 model on a list of utterance strings.

    Returns one dict per utterance, e.g. [{"fear": 0.82, "nervousness": 0.61}, ...]
    Ready to pass directly into EmotionalTrajectoryTracker.add_turn().
    """
    if not USE_REAL_MODEL:
        raise RuntimeError("Model not loaded. Set USE_REAL_MODEL=True and re-run Section 4.")

    import torch
    all_results = []

    for i in range(0, len(utterances), batch_size):
        batch = utterances[i : i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            logits = model(**encoded).logits
            probs = torch.sigmoid(logits).cpu()

        for row in probs:
            emotions = {
                GOEMOTION_LABELS[j]: float(row[j])
                for j in range(len(GOEMOTION_LABELS))
                if float(row[j]) >= threshold
            }
            # Guarantee at least top-1 if nothing clears threshold
            if not emotions:
                top_idx = int(row.argmax())
                emotions = {GOEMOTION_LABELS[top_idx]: float(row[top_idx])}
            all_results.append(emotions)

    return all_results


if USE_REAL_MODEL:
    # ── Quick sanity check ────────────────────────────────────────────────
    test_utterances = [
        "I'm really stressed about my exam tomorrow.",
        "I studied but I feel like I'm going to mess everything up.",
        "My parents are expecting too much.",
    ]
    test_predictions = predict_emotions(test_utterances)

    print("Module 1 predictions:")
    for utt, pred in zip(test_utterances, test_predictions):
        pred_str = ", ".join(f"{k}({v:.2f})" for k, v in sorted(pred.items(), key=lambda x: -x[1]))
        print(f"  [{utt[:55]}...]")
        print(f"    → {pred_str}")
else:
    print("Skipping — USE_REAL_MODEL is False.")


Module 1 predictions:
  [I'm really stressed about my exam tomorrow....]
    → fear(0.56), sadness(0.44)
  [I studied but I feel like I'm going to mess everything ...]
    → sadness(0.47), fear(0.47)
  [My parents are expecting too much....]
    → joy(0.25)


### 4.2 `run_full_pipeline()` — End-to-end: text → signals → conditioning prompt

This chains Module 1 → Module 2 in one call. Given a list of utterance strings, it returns the per-turn predictions, the full `TrajectorySignals`, and the conditioning prompt — everything Module 3 needs to generate a paired response.


In [26]:
def run_full_pipeline(
    conversation: List[str],
    threshold: float = INFERENCE_THRESHOLD,
) -> Tuple[List[Dict[str, float]], TrajectorySignals, str]:
    """
    End-to-end: raw text → Module 1 predictions → Module 2 signals → conditioning prompt.

    Returns
    -------
    (predictions, signals, conditioning_prompt)
    """
    predictions = predict_emotions(conversation, threshold=threshold)

    tracker = EmotionalTrajectoryTracker()
    for turn_emotions in predictions:
        tracker.add_turn(turn_emotions)
    signals = tracker.compute()

    conditioning_prompt = build_conditioning_prompt(signals)
    return predictions, signals, conditioning_prompt


if USE_REAL_MODEL:
    conversation = [
        "I'm really stressed about my exam tomorrow.",
        "I studied but I feel like I'm going to mess everything up.",
        "My parents are expecting too much.",
    ]

    predictions, signals, cond_prompt = run_full_pipeline(conversation)

    print("─" * 60)
    print("  FULL PIPELINE OUTPUT")
    print("─" * 60)
    print(f"\nTrajectory  : {' → '.join(signals.emotion_sequence)}")
    print(f"Dominant    : {signals.dominant_state}")
    print(f"Momentum    : {signals.emotional_momentum:+.3f}")
    print(f"Volatility  : {signals.volatility_index:.3f}")
    print(f"Escalation  : {signals.escalation_score:+.3f}")
    print(f"\nConditioning prompt:")
    print(cond_prompt)
else:
    print("Skipping — USE_REAL_MODEL is False.")


────────────────────────────────────────────────────────────
  FULL PIPELINE OUTPUT
────────────────────────────────────────────────────────────

Trajectory  : Fear → Sadness → Joy
Dominant    : Joy
Momentum    : +1.000
Volatility  : 0.222
Escalation  : -0.799

Conditioning prompt:
[EMOTIONAL CONTEXT — Module 2 output]
User emotional trajectory: de-escalating fear → sadness → joy.
Dominant state: Joy. Momentum: +1.000. Volatility: moderate (0.222). Escalation: -0.799.
Recommended tone: reinforcing + warm encouragement.
[END EMOTIONAL CONTEXT]


## 5. Response Generator — Baseline vs Conditioned

**The core experimental design:** for every conversation, we produce **two** responses:

1. **Baseline** — generic helpful assistant, no emotional awareness
2. **Conditioned** — same LLM, but prepended with the conditioning prompt from Module 2

If Module 2's signals are useful, the conditioned response should score higher on empathy/appropriateness/helpfulness.

**Two modes:**
- **Simulated** (default, runs instantly) — pre-written example pairs for demonstration
- **Live API** — real OpenAI calls; requires `openai` package + API key


In [27]:
BASELINE_SYSTEM_PROMPT = (
    "You are a helpful conversational assistant. "
    "Respond naturally to what the user says."
)

CONDITIONED_SYSTEM_PREFIX = (
    "You are an emotionally intelligent conversational assistant. "
    "Use the emotional context below to calibrate your response.\n\n"
    "{conditioning_prompt}\n\n"
    "Respond with appropriate empathy, tone, and support."
)


def generate_via_api(user_turn: str, conditioning_prompt: str,
                     api_key: str, model_name: str = "gpt-4o-mini") -> Tuple[str, str]:
    """
    Generate real responses via OpenAI API.
    Returns (baseline_response, conditioned_response).
    """
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    # Baseline: generic assistant
    baseline = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": BASELINE_SYSTEM_PROMPT},
            {"role": "user", "content": user_turn},
        ]
    ).choices[0].message.content

    # Conditioned: same LLM + Module 2 conditioning prompt prepended
    system_conditioned = CONDITIONED_SYSTEM_PREFIX.format(
        conditioning_prompt=conditioning_prompt
    )
    conditioned = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_conditioned},
            {"role": "user", "content": user_turn},
        ]
    ).choices[0].message.content

    return baseline, conditioned


print("Response generators defined.")


Response generators defined.


## 6. Build Evaluation Records from Raw Text (new in this version)

This is the **main integration point** between the real Module 1 model and the evaluation pipeline.

Given a list of conversations — where each conversation is a list of utterance strings — this builds a record per conversation containing:
- The trajectory signals (from Module 2)
- The conditioning prompt (built from those signals)
- A baseline response + conditioned response (from a live API call, or manually added)

This replaces the hardcoded `SIMULATED_PAIRS` approach in the previous notebook version.


In [28]:
def build_records_from_text(
    conversations: List[List[str]],
    *,
    api_key: Optional[str] = None,
    model_name: str = "gpt-4o-mini",
    threshold: float = INFERENCE_THRESHOLD,
) -> List[Dict]:
    """
    Build Module 3 evaluation records from raw text conversations.

    Parameters
    ----------
    conversations : list of list of str
        Each inner list is a full conversation (ordered turns as strings).
    api_key : str, optional
        If provided, generates baseline + conditioned responses via OpenAI.
        If None, leaves response fields as placeholders to fill in manually.
    model_name : str
        OpenAI model to use when api_key is set.
    threshold : float
        Sigmoid threshold for Module 1 predictions.

    Returns
    -------
    list of dicts — one per conversation — compatible with AutomatedEvaluator.
    """
    if not USE_REAL_MODEL:
        raise RuntimeError(
            "Need the real Module 1 model to build records from text. "
            "Set USE_REAL_MODEL=True and re-run Section 4."
        )

    records = []
    for i, conv in enumerate(conversations):
        predictions, signals, cond_prompt = run_full_pipeline(conv, threshold=threshold)
        user_last_turn = conv[-1]

        if api_key:
            baseline, conditioned = generate_via_api(
                user_turn=user_last_turn,
                conditioning_prompt=cond_prompt,
                api_key=api_key,
                model_name=model_name,
            )
        else:
            baseline = "[FILL IN BASELINE RESPONSE]"
            conditioned = "[FILL IN CONDITIONED RESPONSE]"

        records.append({
            "conversation_id": f"conv_{i+1:03d}",
            "conversation_turns": conv,
            "user_last_turn": user_last_turn,
            "predictions": predictions,
            "signals": signals,
            "conditioning_prompt": cond_prompt,
            "baseline_response": baseline,
            "conditioned_response": conditioned,
        })

    return records


print("build_records_from_text() defined.")


build_records_from_text() defined.


## 7. Simulated Pairs (Fallback — No Model Required)

If you can't load the Module 1 model (e.g. running a quick demo, or a CI pipeline), these pre-written pairs let you test the evaluation end-to-end without the model.

Each entry contains a pre-computed emotion sequence, hand-written baseline and conditioned responses, and enough context to run the scoring pipeline.


In [29]:
SIMULATED_PAIRS = [
    {
        "conversation": [
            {"Anxiety": 0.72, "Stress": 0.81},
            {"Anxiety": 0.85, "Self-doubt": 0.76},
            {"Pressure": 0.89, "Frustration": 0.67},
        ],
        "user_last_turn": "My parents are expecting too much from me right now.",
        "baseline_response": (
            "That sounds like a lot. Have you tried talking to your parents about your expectations?"
        ),
        "conditioned_response": (
            "It sounds like you're carrying a lot right now — the anxiety about your exam "
            "and feeling weighed down by your parents' expectations on top of that. "
            "That's genuinely exhausting. You don't have to have everything figured out. "
            "What feels like the heaviest part right now?"
        ),
    },
    {
        "conversation": [
            {"Anger": 0.90, "Resentment": 0.70},
            {"Anger": 0.75, "Frustration": 0.60},
            {"Sadness": 0.65, "Disappointment": 0.55},
        ],
        "user_last_turn": "I just feel like nobody listens to me at work.",
        "baseline_response": (
            "That's frustrating. Maybe try scheduling a one-on-one with your manager."
        ),
        "conditioned_response": (
            "I can hear how drained you are — moving from anger to just feeling unheard "
            "and sad is its own kind of exhaustion. It's not just about work logistics; "
            "it feels like a deeper sense that your perspective doesn't matter to the people around you. "
            "That deserves to be acknowledged. What do you most wish they understood about you?"
        ),
    },
    {
        "conversation": [
            {"Joy": 0.85, "Excitement": 0.70},
            {"Joy": 0.60, "Nervousness": 0.45},
            {"Anxiety": 0.72, "Disappointment": 0.55},
            {"Sadness": 0.80, "Grief": 0.40},
        ],
        "user_last_turn": "I didn't get the job offer I was hoping for.",
        "baseline_response": (
            "Sorry to hear that. There will be other opportunities!"
        ),
        "conditioned_response": (
            "Oh, that's really hard — especially after going in hopeful and excited. "
            "It makes sense that the drop feels so sharp right now. "
            "Grieving a missed opportunity is real. Give yourself a moment before jumping to next steps. "
            "How are you feeling about everything else right now?"
        ),
    },
]


def build_records_from_simulated(pairs: List[Dict] = None) -> List[Dict]:
    """Build evaluation records from the hand-written SIMULATED_PAIRS."""
    pairs = pairs or SIMULATED_PAIRS
    records = []
    for i, item in enumerate(pairs):
        tracker = EmotionalTrajectoryTracker()
        for turn in item["conversation"]:
            tracker.add_turn(turn)
        signals = tracker.compute()
        conditioning_prompt = build_conditioning_prompt(signals)

        records.append({
            "conversation_id": f"sim_{i+1:03d}",
            "conversation_turns": None,  # simulated pairs don't have raw text
            "user_last_turn": item["user_last_turn"],
            "predictions": item["conversation"],
            "signals": signals,
            "conditioning_prompt": conditioning_prompt,
            "baseline_response": item["baseline_response"],
            "conditioned_response": item["conditioned_response"],
        })
    return records


print(f"Loaded {len(SIMULATED_PAIRS)} simulated conversation pairs.")


Loaded 3 simulated conversation pairs.


## 8. Automated Evaluation Metrics

Three complementary metrics — each catches something the others miss.

### Metric 1: BERTScore F1 (30% weight)
Semantic similarity between a response and a "gold" empathetic reference. Uses contextual BERT embeddings. Falls back to Jaccard token overlap if `bert-score` isn't installed.

### Metric 2: Empathy Score (50% weight — the most important)
Uses zero-shot NLI with `facebook/bart-large-mnli` to check if the response entails an empathetic stance. Directly tests the proposal's hypothesis.

### Metric 3: Specificity (20% weight)
Rule-based check for whether the response reflects Module 2's specific emotional context (dominant state, trajectory direction, intensity).


In [30]:
@dataclass
class ResponseScores:
    """Per-conversation evaluation scores."""
    conversation_id: str
    bertscore_baseline: float = 0.0
    bertscore_conditioned: float = 0.0
    empathy_baseline: float = 0.0
    empathy_conditioned: float = 0.0
    specificity_baseline: float = 0.0
    specificity_conditioned: float = 0.0
    length_baseline: int = 0
    length_conditioned: int = 0
    overall_baseline: float = 0.0
    overall_conditioned: float = 0.0
    improvement_pct: float = 0.0


class AutomatedEvaluator:
    """Three-metric evaluator: BERTScore + Empathy + Specificity."""

    def __init__(self):
        self._bertscore_model = None
        self._nli_pipeline = None
        self._loaded = False

    def _load_models(self):
        if self._loaded:
            return
        print("Loading evaluation models (first run only)...")

        try:
            from bert_score import BERTScorer
            self._bertscore_model = BERTScorer(lang="en", rescale_with_baseline=True)
        except ImportError:
            print("  [!] bert-score not installed — using Jaccard fallback.")

        try:
            from transformers import pipeline
            self._nli_pipeline = pipeline(
                "zero-shot-classification",
                model="facebook/bart-large-mnli",
                device=-1,
            )
        except ImportError:
            print("  [!] transformers not installed — using keyword fallback.")

        self._loaded = True

    def _bertscore(self, response: str, reference: str) -> float:
        if self._bertscore_model is not None:
            _, _, F = self._bertscore_model.score([response], [reference])
            return float(F[0])
        ref_tokens = set(reference.lower().split())
        res_tokens = set(response.lower().split())
        return len(ref_tokens & res_tokens) / len(ref_tokens | res_tokens) if ref_tokens else 0.0

    def _empathy_score(self, response: str, dominant_emotion: str) -> float:
        if self._nli_pipeline is not None:
            # Zero-shot classification: the pipeline substitutes each candidate
            # label into the {} placeholder. We frame it as "empathetic" vs
            # "dismissive" — higher entailment with "empathetic" = more empathy.
            result = self._nli_pipeline(
                response,
                candidate_labels=["empathetic and validating", "dismissive or cold"],
                hypothesis_template=(
                    f"This response to a user expressing {dominant_emotion.lower()} "
                    f"is {{}}."
                ),
            )
            scores = dict(zip(result["labels"], result["scores"]))
            return float(scores.get("empathetic and validating", 0.0))
        empathy_keywords = {"understand", "hear", "feel", "difficult", "hard", "support",
                            "acknowledge", "sense", "exhausting", "sounds like", "that's",
                            "valid", "matter", "grieve", "sorry"}
        words = set(response.lower().split())
        return min(1.0, len(words & empathy_keywords) / 5.0)

    @staticmethod
    def _specificity(response: str, signals: TrajectorySignals) -> float:
        resp_lower = response.lower()
        score = 0.0

        if signals.dominant_state.lower() in resp_lower:
            score += 0.4
        elif any(w in resp_lower for w in ["anxious", "stress", "fear", "anger",
                                            "sad", "joy", "disgust", "contempt"]):
            score += 0.2

        if signals.escalation_score > 0.1:
            if any(w in resp_lower for w in ["more", "building", "growing", "heavy", "exhausting"]):
                score += 0.3
        elif signals.escalation_score < -0.1:
            if any(w in resp_lower for w in ["better", "improving", "progress", "calmer"]):
                score += 0.3
        else:
            score += 0.15

        if signals.volatility_index > 0.4:
            if any(w in resp_lower for w in ["lot", "much", "overwhelming", "intense", "hard"]):
                score += 0.3

        return min(1.0, score)

    def evaluate(self, records: List[Dict],
                 gold_references: Optional[List[str]] = None) -> List[ResponseScores]:
        self._load_models()

        results = []
        for i, rec in enumerate(tqdm(records, desc="Evaluating")):
            signals = rec["signals"]
            baseline = rec["baseline_response"]
            conditioned = rec["conditioned_response"]
            reference = gold_references[i] if gold_references else conditioned

            bs_b = self._bertscore(baseline, reference)
            bs_c = self._bertscore(conditioned, reference)
            emp_b = self._empathy_score(baseline, signals.dominant_state)
            emp_c = self._empathy_score(conditioned, signals.dominant_state)
            spec_b = self._specificity(baseline, signals)
            spec_c = self._specificity(conditioned, signals)

            overall_b = 0.5 * emp_b + 0.3 * bs_b + 0.2 * spec_b
            overall_c = 0.5 * emp_c + 0.3 * bs_c + 0.2 * spec_c
            improvement = ((overall_c - overall_b) / overall_b * 100) if overall_b > 0 else 0.0

            results.append(ResponseScores(
                conversation_id=rec["conversation_id"],
                bertscore_baseline=round(bs_b, 4),
                bertscore_conditioned=round(bs_c, 4),
                empathy_baseline=round(emp_b, 4),
                empathy_conditioned=round(emp_c, 4),
                specificity_baseline=round(spec_b, 4),
                specificity_conditioned=round(spec_c, 4),
                length_baseline=len(baseline.split()),
                length_conditioned=len(conditioned.split()),
                overall_baseline=round(overall_b, 4),
                overall_conditioned=round(overall_c, 4),
                improvement_pct=round(improvement, 2),
            ))
        return results


print("AutomatedEvaluator defined.")


AutomatedEvaluator defined.


## 9. Human Rater Export

Exports an anonymised A/B CSV for 20–30 human evaluators. Raters score each response on empathy, appropriateness, and helpfulness (1–5 each). Order is randomised per conversation to prevent bias.


In [31]:
def export_human_rater_csv(records: List[Dict], path: str = "human_rater_sheet.csv") -> None:
    """Export anonymised A/B response pairs for human rating."""
    import random
    rows = []
    for rec in records:
        pairs = [("A", rec["baseline_response"]),
                 ("B", rec["conditioned_response"])]
        random.shuffle(pairs)

        for label, response in pairs:
            rows.append({
                "conversation_id": rec["conversation_id"],
                "response_label": label,
                "user_last_turn": rec["user_last_turn"],
                "response_text": response,
                "empathy_score": "",
                "appropriateness_score": "",
                "helpfulness_score": "",
                "notes": "",
            })

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"Human rater CSV saved → {path}  ({len(rows)} rows for {len(records)} conversations)")


def load_human_ratings(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    for col in ["empathy_score", "appropriateness_score", "helpfulness_score"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["composite"] = df[["empathy_score", "appropriateness_score",
                          "helpfulness_score"]].mean(axis=1)
    return df


print("Human rater export functions defined.")


Human rater export functions defined.


## 10. Reporting

Prints per-conversation scores, aggregate means, mean improvement %, and whether the proposal's ≥ 20% target was hit.


In [32]:
def print_evaluation_report(scores: List[ResponseScores]) -> None:
    print("\n" + "═" * 75)
    print("  MODULE 3 — RESPONSE EVALUATION REPORT")
    print("═" * 75)

    df = pd.DataFrame([vars(s) for s in scores])

    print("\nPer-Conversation Scores (Baseline → Conditioned)\n")
    print(f"{'Conv':<12}{'BERTScore':>18}{'Empathy':>18}{'Specificity':>18}{'Overall':>18}{'Δ%':>8}")
    print("-" * 92)
    for s in scores:
        print(f"{s.conversation_id:<12}"
              f"  {s.bertscore_baseline:.3f} → {s.bertscore_conditioned:.3f}"
              f"  {s.empathy_baseline:.3f} → {s.empathy_conditioned:.3f}"
              f"  {s.specificity_baseline:.3f} → {s.specificity_conditioned:.3f}"
              f"  {s.overall_baseline:.3f} → {s.overall_conditioned:.3f}"
              f"  {s.improvement_pct:>+.1f}%")

    print("\n" + "─" * 75)
    print("AGGREGATE (mean across all conversations)\n")
    metrics = {
        "BERTScore": ("bertscore_baseline", "bertscore_conditioned"),
        "Empathy": ("empathy_baseline", "empathy_conditioned"),
        "Specificity": ("specificity_baseline", "specificity_conditioned"),
        "Overall": ("overall_baseline", "overall_conditioned"),
    }
    for label, (col_b, col_c) in metrics.items():
        mean_b = df[col_b].mean()
        mean_c = df[col_c].mean()
        delta = mean_c - mean_b
        pct = (delta / mean_b * 100) if mean_b > 0 else 0.0
        bar = "▓" * int(abs(pct) / 5) if abs(pct) > 0 else "·"
        print(f"  {label:<14}: {mean_b:.4f} → {mean_c:.4f}  "
              f"(Δ {delta:+.4f}  |  {pct:+.1f}%  {bar})")

    avg_improvement = df["improvement_pct"].mean()
    print(f"\n  Mean overall improvement : {avg_improvement:+.2f}%")

    target = 20.0
    status = ("✅ HYPOTHESIS SUPPORTED" if avg_improvement >= target
              else f"❌ Below {target}% target")
    print(f"  Target (≥ {target}%)         : {status}")

    print("\n  Response lengths (words):")
    print(f"    Baseline avg    : {df['length_baseline'].mean():.1f}")
    print(f"    Conditioned avg : {df['length_conditioned'].mean():.1f}")
    print("═" * 75)


def export_results_csv(scores: List[ResponseScores], path: str = "evaluation_results.csv") -> None:
    pd.DataFrame([vars(s) for s in scores]).to_csv(path, index=False)
    print(f"Results saved → {path}")


print("Reporting functions defined.")


Reporting functions defined.


## 11. Run on Real Conversations (Using the Loaded Module 1 Model)

This is the headline demo of this updated notebook. We pass raw-text conversations through the full pipeline: Module 1 predicts emotions → Module 2 computes trajectories → Module 3 builds the conditioning prompt → (optionally) an LLM generates both responses → automated scoring.

**Define your test conversations below.** Each conversation is a list of utterance strings — real dialogue, not pre-scored labels.


In [33]:
# ── Real conversations: raw text, one string per turn ─────────────────────────
REAL_TEST_CONVERSATIONS = [
    # Conversation 1: Exam stress spiral (escalating anxiety)
    [
        "I'm really stressed about my exam tomorrow.",
        "I studied but I feel like I'm going to mess everything up.",
        "My parents are expecting too much from me right now.",
    ],
    # Conversation 2: Work frustration into resignation
    [
        "I keep getting cut off in meetings today.",
        "Nobody seems to take my ideas seriously.",
        "I just feel like nobody listens to me at work.",
    ],
    # Conversation 3: Hope → disappointment (sharp fall)
    [
        "I have my final interview tomorrow, I'm so excited!",
        "I hope I don't mess this up, I really want this role.",
        "I didn't get the job offer I was hoping for.",
    ],
]

# ── Set to your OpenAI key to generate real responses; leave None for manual ──
OPENAI_API_KEY = None  # e.g. "sk-..."
OPENAI_MODEL = "gpt-4o-mini"

if USE_REAL_MODEL:
    real_records = build_records_from_text(
        REAL_TEST_CONVERSATIONS,
        api_key=OPENAI_API_KEY,
        model_name=OPENAI_MODEL,
    )

    for rec in real_records:
        print(f"\n{'─' * 70}")
        print(f"{rec['conversation_id']}")
        print(f"{'─' * 70}")
        print(f"Conversation   : {rec['conversation_turns']}")
        print(f"Trajectory     : {' → '.join(rec['signals'].emotion_sequence)}")
        print(f"Escalation     : {rec['signals'].escalation_score:+.3f}")
        print(f"Volatility     : {rec['signals'].volatility_index:.3f}")
        print(f"\nConditioning prompt:")
        print(rec['conditioning_prompt'])
        print(f"\nBaseline response:    {rec['baseline_response'][:150]}...")
        print(f"Conditioned response: {rec['conditioned_response'][:150]}...")
else:
    print("Set USE_REAL_MODEL=True in Section 4 to run this cell.")
    real_records = []



──────────────────────────────────────────────────────────────────────
conv_001
──────────────────────────────────────────────────────────────────────
Conversation   : ["I'm really stressed about my exam tomorrow.", "I studied but I feel like I'm going to mess everything up.", 'My parents are expecting too much from me right now.']
Trajectory     : Fear → Sadness → Joy
Escalation     : -0.799
Volatility     : 0.165

Conditioning prompt:
[EMOTIONAL CONTEXT — Module 2 output]
User emotional trajectory: de-escalating fear → sadness → joy.
Dominant state: Joy. Momentum: +1.000. Volatility: moderate (0.165). Escalation: -0.799.
Recommended tone: reinforcing + warm encouragement.
[END EMOTIONAL CONTEXT]

Baseline response:    [FILL IN BASELINE RESPONSE]...
Conditioned response: [FILL IN CONDITIONED RESPONSE]...

──────────────────────────────────────────────────────────────────────
conv_002
──────────────────────────────────────────────────────────────────────
Conversation   : ['I keep gett

## 12. Run Evaluation

Runs the three-metric evaluation on whichever records you built — real (Section 11) or simulated (Section 7).


In [34]:
# Choose which records to evaluate. Real records require the model + (optionally) an API key.
# If real_records is empty OR baseline/conditioned responses are placeholders,
# fall back to simulated pairs.

def _has_real_responses(recs):
    if not recs:
        return False
    for r in recs:
        if r["baseline_response"].startswith("[FILL IN") or r["conditioned_response"].startswith("[FILL IN"):
            return False
    return True


if USE_REAL_MODEL and _has_real_responses(real_records):
    records_to_evaluate = real_records
    print(f"Using {len(records_to_evaluate)} real records (with live LLM responses).")
elif USE_REAL_MODEL and real_records:
    print("Real records have placeholder responses — falling back to simulated pairs.")
    records_to_evaluate = build_records_from_simulated()
else:
    records_to_evaluate = build_records_from_simulated()
    print(f"Using {len(records_to_evaluate)} simulated pairs.")


Real records have placeholder responses — falling back to simulated pairs.


In [35]:
# ── Run automated evaluation ──────────────────────────────────────────────────
# First call downloads bert-score + bart-large-mnli models (~1.5 GB)
evaluator = AutomatedEvaluator()
scores = evaluator.evaluate(records_to_evaluate)


Loading evaluation models (first run only)...


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 10293.45it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Evaluating: 100%|██████████| 3/3 [00:07<00:00,  2.49s/it]


In [36]:
# ── Print final report ────────────────────────────────────────────────────────
print_evaluation_report(scores)



═══════════════════════════════════════════════════════════════════════════
  MODULE 3 — RESPONSE EVALUATION REPORT
═══════════════════════════════════════════════════════════════════════════

Per-Conversation Scores (Baseline → Conditioned)

Conv                 BERTScore           Empathy       Specificity           Overall      Δ%
--------------------------------------------------------------------------------------------
sim_001       0.314 → 1.000  0.285 → 0.925  0.150 → 0.150  0.267 → 0.793  +197.2%
sim_002       0.153 → 1.000  0.125 → 0.926  0.000 → 0.400  0.108 → 0.843  +676.9%
sim_003       0.211 → 1.000  0.754 → 0.977  0.000 → 0.000  0.440 → 0.789  +79.1%

───────────────────────────────────────────────────────────────────────────
AGGREGATE (mean across all conversations)

  BERTScore     : 0.2260 → 1.0000  (Δ +0.7740  |  +342.4%  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓)
  Empathy       : 0.3880 → 0.9428  (Δ +0.5548  |  +143.0%  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

In [37]:
# ── Export results + human rater CSV ──────────────────────────────────────────
export_results_csv(scores, "evaluation_results.csv")
export_human_rater_csv(records_to_evaluate, "human_rater_sheet.csv")
print("\nDone. Share human_rater_sheet.csv with your 20–30 evaluators.")


Results saved → evaluation_results.csv
Human rater CSV saved → human_rater_sheet.csv  (6 rows for 3 conversations)

Done. Share human_rater_sheet.csv with your 20–30 evaluators.


## 13. Loading Human Ratings Back (After Collection)

Once raters have filled in `human_rater_sheet.csv`, reload and aggregate.


In [38]:
# Example — uncomment after human ratings are collected

# human_df = load_human_ratings("human_rater_sheet_completed.csv")
#
# summary = human_df.groupby("response_label")[[
#     "empathy_score", "appropriateness_score", "helpfulness_score", "composite"
# ]].mean()
# print(summary)


## Summary

**What this notebook accomplishes:**

| Step | Outcome |
|---|---|
| 1. Load Module 1 RoBERTa from safetensors | Can run real emotion detection on raw text |
| 2. Replicate Module 2 | Can compute trajectory signals + conditioning prompts |
| 3. `run_full_pipeline()` | Chains Module 1 → Module 2 on any list of utterance strings |
| 4. `build_records_from_text()` | Turns raw conversations into evaluation records |
| 5. `build_records_from_simulated()` | Fallback mode for testing without the model |
| 6. Automated evaluation | BERTScore + Empathy (NLI) + Specificity composite |
| 7. Human rater export | Anonymised CSV for 20–30 evaluators |
| 8. Report | Tests the proposal's ≥ 20% improvement hypothesis |

**Key improvement over the previous version:**

The previous Module 3 notebook required manually specifying emotion scores for each turn. This version accepts raw text conversations and runs them through the actual fine-tuned RoBERTa model — matching the pattern established in the Module 2 v2 notebook. This is what the full end-to-end evaluation needs.

**Next steps:**

1. Provide an OpenAI API key in Section 11 to generate real baseline and conditioned responses.
2. Expand `REAL_TEST_CONVERSATIONS` to 50–100 conversations from DailyDialog held-out split.
3. Deploy `human_rater_sheet.csv` to 20–30 evaluators.
4. Correlate automated scores with human scores to validate the automated metric.
